# MNQ ORB - Comparatif propre `nominal` vs `sizing_3state` retenu

Ce notebook compare le baseline `nominal` au **3-state retenu**:

- `low = 0.50x`
- `mid = 1.00x`
- `high = 0.25x`

La comparaison reste saine:

- même signal ORB,
- mêmes entrées / sorties,
- même stop / target,
- même logique de coûts,
- seule l'exposition varie par bucket.


In [1]:
import json
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent

if not (ROOT / "pyproject.toml").exists():
    raise RuntimeError("Impossible de retrouver la racine du repo depuis le notebook.")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import Markdown, display
from plotly.subplots import make_subplots

from src.analytics.metrics import compute_metrics
from src.analytics.mnq_orb_prop_survivability_campaign import _rebuild_daily_results_from_trades
from src.analytics.mnq_orb_regime_filter_sizing_campaign import _scale_nominal_trades_by_multiplier

pd.set_option("display.max_columns", 300)
pd.set_option("display.width", 240)


def fmt_money(value):
    if value is None or pd.isna(value):
        return "n/a"
    return f"{float(value):,.1f} USD"


def fmt_float(value, digits=3):
    if value is None or pd.isna(value):
        return "n/a"
    return f"{float(value):.{digits}f}"


def build_curve_from_daily(daily, initial_balance):
    out = daily.copy()
    out["session_date"] = pd.to_datetime(out["session_date"], errors="coerce")
    out = out.sort_values("session_date").reset_index(drop=True)
    out["daily_pnl_usd"] = pd.to_numeric(out["daily_pnl_usd"], errors="coerce").fillna(0.0)
    out["equity"] = initial_balance + out["daily_pnl_usd"].cumsum()
    out["peak_equity"] = out["equity"].cummax()
    out["drawdown_usd"] = out["equity"] - out["peak_equity"]
    out["drawdown_pct"] = (out["equity"] / out["peak_equity"] - 1.0) * 100.0
    return out


def rebase_oos_curve(curve_df, start_date, initial_balance):
    curve = curve_df.loc[curve_df["session_date"] >= pd.Timestamp(start_date)].copy()
    curve = curve.sort_values("session_date").reset_index(drop=True)
    curve["equity"] = initial_balance + curve["daily_pnl_usd"].cumsum()
    curve["peak_equity"] = curve["equity"].cummax()
    curve["drawdown_usd"] = curve["equity"] - curve["peak_equity"]
    curve["drawdown_pct"] = (curve["equity"] / curve["peak_equity"] - 1.0) * 100.0
    return curve


def scope_metrics(trades, sessions, initial_capital):
    metrics = compute_metrics(trades, session_dates=sessions, initial_capital=initial_capital)
    return {
        "net_pnl": float(metrics.get("cumulative_pnl", 0.0)),
        "sharpe": float(metrics.get("sharpe_ratio", 0.0)),
        "profit_factor": float(metrics.get("profit_factor", 0.0)),
        "max_drawdown": float(metrics.get("max_drawdown", 0.0)),
        "n_trades": int(metrics.get("n_trades", 0)),
        "pct_days_traded": float(metrics.get("percent_of_days_traded", 0.0)),
        "worst_day": float(metrics.get("worst_day", 0.0)),
        "win_rate": float(metrics.get("win_rate", 0.0)),
    }


In [2]:
REGIME_EXPORT_ROOT = ROOT / r"data\exports\mnq_orb_regime_filter_sizing_20260325_150405"
VARIANT_NAME = "sizing_3state_realized_vol_ratio_15_60"
BASELINE_NAME = "nominal"
HIGH_BUCKET_MULTIPLIER = 0.25
LOW_BUCKET_MULTIPLIER = 0.50
MID_BUCKET_MULTIPLIER = 1.00
INITIAL_BALANCE_USD = 50_000.0

required_paths = {
    "regime_export_root": REGIME_EXPORT_ROOT,
    "summary_variants": REGIME_EXPORT_ROOT / "summary_variants.csv",
    "baseline_metrics": REGIME_EXPORT_ROOT / "variants" / BASELINE_NAME / "metrics_by_scope.csv",
    "baseline_daily": REGIME_EXPORT_ROOT / "variants" / BASELINE_NAME / "daily_results.csv",
    "baseline_trades": REGIME_EXPORT_ROOT / "variants" / BASELINE_NAME / "trades.csv",
    "variant_controls": REGIME_EXPORT_ROOT / "variants" / VARIANT_NAME / "controls.csv",
    "regime_mapping": REGIME_EXPORT_ROOT / "regime_state_mappings.csv",
}

missing = [name for name, path in required_paths.items() if not path.exists()]
if missing:
    raise FileNotFoundError(f"Fichiers manquants pour le notebook: {missing}")

print("REGIME_EXPORT_ROOT =", REGIME_EXPORT_ROOT)
print("BASELINE_NAME      =", BASELINE_NAME)
print("VARIANT_NAME       =", VARIANT_NAME)
print("HIGH_MULTIPLIER    =", HIGH_BUCKET_MULTIPLIER)


REGIME_EXPORT_ROOT = C:\Data\Perso\algo-trading-intraday-research\data\exports\mnq_orb_regime_filter_sizing_20260325_150405
BASELINE_NAME      = nominal
VARIANT_NAME       = sizing_3state_realized_vol_ratio_15_60
HIGH_MULTIPLIER    = 0.25


In [3]:
regime_metadata = json.loads((REGIME_EXPORT_ROOT / "run_metadata.json").read_text(encoding="utf-8"))
summary_variants = pd.read_csv(REGIME_EXPORT_ROOT / "summary_variants.csv")
baseline_metrics = pd.read_csv(REGIME_EXPORT_ROOT / "variants" / BASELINE_NAME / "metrics_by_scope.csv")
baseline_daily = pd.read_csv(REGIME_EXPORT_ROOT / "variants" / BASELINE_NAME / "daily_results.csv", parse_dates=["session_date"])
baseline_trades = pd.read_csv(
    REGIME_EXPORT_ROOT / "variants" / BASELINE_NAME / "trades.csv",
    parse_dates=["session_date", "entry_time", "exit_time"],
)
variant_controls = pd.read_csv(
    REGIME_EXPORT_ROOT / "variants" / VARIANT_NAME / "controls.csv",
    parse_dates=["session_date"],
)
regime_mapping = pd.read_csv(REGIME_EXPORT_ROOT / "regime_state_mappings.csv")

baseline_trades["session_date"] = pd.to_datetime(baseline_trades["session_date"], errors="coerce")
baseline_trades["entry_time"] = pd.to_datetime(baseline_trades["entry_time"], errors="coerce", utc=True)
baseline_trades["exit_time"] = pd.to_datetime(baseline_trades["exit_time"], errors="coerce", utc=True)
variant_controls["session_date"] = pd.to_datetime(variant_controls["session_date"], errors="coerce")

baseline_row = summary_variants.loc[summary_variants["variant_name"] == BASELINE_NAME].iloc[0]
baseline_config = regime_metadata["spec"]["baseline"]
initial_capital = float(baseline_config["account_size_usd"])
base_risk_pct = float(baseline_config["risk_per_trade_pct"])
all_sessions = pd.to_datetime(baseline_daily["session_date"], errors="coerce").dt.date.tolist()
is_sessions = pd.to_datetime(variant_controls.loc[variant_controls["phase"] == "is", "session_date"], errors="coerce").dt.date.tolist()
oos_sessions = pd.to_datetime(variant_controls.loc[variant_controls["phase"] == "oos", "session_date"], errors="coerce").dt.date.tolist()
oos_start_date = pd.to_datetime(variant_controls.loc[variant_controls["phase"] == "oos", "session_date"].min())

bucket_map = (
    regime_mapping.loc[
        (regime_mapping["variant_name"] == VARIANT_NAME)
        & (regime_mapping["feature_name"] == "realized_vol_ratio_15_60"),
        ["bucket_label", "bucket_position", "lower_bound", "upper_bound", "risk_multiplier", "is_composite_score", "oos_n_obs", "oos_net_pnl", "oos_sharpe", "oos_max_drawdown"],
    ]
    .drop_duplicates()
    .sort_values("bucket_position")
    .reset_index(drop=True)
)

variant_controls["bucket_label"] = variant_controls["bucket_label"].astype(str)
variant_controls["risk_multiplier"] = variant_controls["bucket_label"].map(
    {
        "low": LOW_BUCKET_MULTIPLIER,
        "mid": MID_BUCKET_MULTIPLIER,
        "high": HIGH_BUCKET_MULTIPLIER,
    }
).fillna(0.0)
variant_controls["skip_trade"] = pd.to_numeric(variant_controls["risk_multiplier"], errors="coerce").fillna(0.0).le(0.0)

variant_trades = _scale_nominal_trades_by_multiplier(
    nominal_trades=baseline_trades,
    controls=variant_controls,
    account_size_usd=initial_capital,
    base_risk_pct=base_risk_pct,
    tick_value_usd=0.5,
    point_value_usd=2.0,
    commission_per_side_usd=1.25,
)
variant_trades["session_date"] = pd.to_datetime(variant_trades["session_date"], errors="coerce")
variant_trades["entry_time"] = pd.to_datetime(variant_trades["entry_time"], errors="coerce", utc=True)
variant_trades["exit_time"] = pd.to_datetime(variant_trades["exit_time"], errors="coerce", utc=True)

variant_daily = _rebuild_daily_results_from_trades(variant_trades, all_sessions=all_sessions, initial_capital=initial_capital)
variant_daily["session_date"] = pd.to_datetime(variant_daily["session_date"], errors="coerce")

variant_metrics = pd.DataFrame(
    [
        {"scope": "overall", **scope_metrics(variant_trades, all_sessions, initial_capital)},
        {"scope": "is", **scope_metrics(variant_trades.loc[pd.to_datetime(variant_trades["session_date"], errors="coerce").dt.date.isin(set(is_sessions))].copy(), is_sessions, initial_capital)},
        {"scope": "oos", **scope_metrics(variant_trades.loc[pd.to_datetime(variant_trades["session_date"], errors="coerce").dt.date.isin(set(oos_sessions))].copy(), oos_sessions, initial_capital)},
    ]
)

variant_row = pd.Series(
    {
        "variant_name": f"sizing_3state_high_{str(HIGH_BUCKET_MULTIPLIER).replace('.', 'p')}",
        "overall_net_pnl": float(variant_metrics.loc[variant_metrics["scope"] == "overall", "net_pnl"].iloc[0]),
        "overall_sharpe": float(variant_metrics.loc[variant_metrics["scope"] == "overall", "sharpe"].iloc[0]),
        "overall_profit_factor": float(variant_metrics.loc[variant_metrics["scope"] == "overall", "profit_factor"].iloc[0]),
        "overall_max_drawdown": float(variant_metrics.loc[variant_metrics["scope"] == "overall", "max_drawdown"].iloc[0]),
        "oos_net_pnl": float(variant_metrics.loc[variant_metrics["scope"] == "oos", "net_pnl"].iloc[0]),
        "oos_sharpe": float(variant_metrics.loc[variant_metrics["scope"] == "oos", "sharpe"].iloc[0]),
        "oos_profit_factor": float(variant_metrics.loc[variant_metrics["scope"] == "oos", "profit_factor"].iloc[0]),
        "oos_max_drawdown": float(variant_metrics.loc[variant_metrics["scope"] == "oos", "max_drawdown"].iloc[0]),
    }
)
variant_row["oos_net_pnl_retention_vs_nominal"] = variant_row["oos_net_pnl"] / float(baseline_row["oos_net_pnl"]) if float(baseline_row["oos_net_pnl"]) != 0 else np.nan
variant_row["oos_sharpe_delta_vs_nominal"] = variant_row["oos_sharpe"] - float(baseline_row["oos_sharpe"])
variant_row["oos_max_drawdown_improvement_vs_nominal"] = abs(float(baseline_row["oos_max_drawdown"])) - abs(float(variant_row["oos_max_drawdown"]))

bucket_map["effective_risk_per_trade_pct"] = bucket_map["bucket_label"].map(
    {"low": LOW_BUCKET_MULTIPLIER, "mid": MID_BUCKET_MULTIPLIER, "high": HIGH_BUCKET_MULTIPLIER}
) * base_risk_pct

baseline_curve = build_curve_from_daily(baseline_daily, INITIAL_BALANCE_USD)
variant_curve = build_curve_from_daily(variant_daily, INITIAL_BALANCE_USD)
baseline_curve_oos = rebase_oos_curve(baseline_daily, oos_start_date, INITIAL_BALANCE_USD)
variant_curve_oos = rebase_oos_curve(variant_daily, oos_start_date, INITIAL_BALANCE_USD)

baseline_trades["trade_key"] = baseline_trades["entry_time"].dt.strftime("%Y-%m-%d %H:%M:%S%z") + "|" + baseline_trades["direction"].astype(str)
variant_trades["trade_key"] = variant_trades["entry_time"].dt.strftime("%Y-%m-%d %H:%M:%S%z") + "|" + variant_trades["direction"].astype(str)

trade_comparison = (
    baseline_trades.rename(
        columns={
            "quantity": "quantity_nominal",
            "net_pnl_usd": "net_pnl_nominal",
            "risk_per_trade_pct": "risk_pct_nominal",
            "actual_risk_usd": "actual_risk_nominal",
            "fees": "fees_nominal",
            "exit_reason": "exit_reason_nominal",
            "exit_time": "exit_time_nominal",
            "stop_price": "stop_price_nominal",
            "target_price": "target_price_nominal",
        }
    )
    .merge(
        variant_trades.rename(
            columns={
                "quantity": "quantity_3state",
                "net_pnl_usd": "net_pnl_3state",
                "risk_per_trade_pct": "risk_pct_3state",
                "actual_risk_usd": "actual_risk_3state",
                "fees": "fees_3state",
                "exit_reason": "exit_reason_3state",
                "exit_time": "exit_time_3state",
                "stop_price": "stop_price_3state",
                "target_price": "target_price_3state",
            }
        )[
            [
                "trade_key",
                "quantity_3state",
                "net_pnl_3state",
                "risk_pct_3state",
                "actual_risk_3state",
                "fees_3state",
                "risk_multiplier",
                "exit_reason_3state",
                "exit_time_3state",
                "stop_price_3state",
                "target_price_3state",
            ]
        ],
        on="trade_key",
        how="outer",
    )
    .sort_values("entry_time")
    .reset_index(drop=True)
)

trade_comparison["same_trade"] = trade_comparison["quantity_nominal"].notna() & trade_comparison["quantity_3state"].notna()
trade_comparison["same_exit_time"] = trade_comparison["exit_time_nominal"].eq(trade_comparison["exit_time_3state"])
trade_comparison["same_exit_reason"] = trade_comparison["exit_reason_nominal"].eq(trade_comparison["exit_reason_3state"])
trade_comparison["same_stop_price"] = np.isclose(trade_comparison["stop_price_nominal"], trade_comparison["stop_price_3state"], equal_nan=True)
trade_comparison["same_target_price"] = np.isclose(trade_comparison["target_price_nominal"], trade_comparison["target_price_3state"], equal_nan=True)
trade_comparison["size_ratio_3state_vs_nominal"] = pd.to_numeric(trade_comparison["quantity_3state"], errors="coerce") / pd.to_numeric(trade_comparison["quantity_nominal"], errors="coerce")
trade_comparison["pnl_ratio_3state_vs_nominal"] = pd.to_numeric(trade_comparison["net_pnl_3state"], errors="coerce") / pd.to_numeric(trade_comparison["net_pnl_nominal"], errors="coerce")
trade_comparison["abs_pnl_ratio_3state_vs_nominal"] = pd.to_numeric(trade_comparison["net_pnl_3state"], errors="coerce").abs() / pd.to_numeric(trade_comparison["net_pnl_nominal"], errors="coerce").abs()

same_trade_rows = trade_comparison.loc[trade_comparison["same_trade"]].copy()
display(Markdown(f"**OOS start date:** `{oos_start_date.date()}`"))


**OOS start date:** `2024-02-23`

## 1. Lecture Rapide

Cette version correspond au 3-state retenu:

- `low = 0.50x`
- `mid = 1.00x`
- `high = 0.25x`


In [4]:
quick_lines = [
    "### Synthese executive",
    f"- Le nominal OOS fait `{fmt_money(baseline_row['oos_net_pnl'])}` avec Sharpe `{fmt_float(baseline_row['oos_sharpe'])}` et maxDD `{fmt_money(baseline_row['oos_max_drawdown'])}`.",
    f"- Le 3-state retenu OOS fait `{fmt_money(variant_row['oos_net_pnl'])}` avec Sharpe `{fmt_float(variant_row['oos_sharpe'])}` et maxDD `{fmt_money(variant_row['oos_max_drawdown'])}`.",
    f"- La variante retenue conserve **{variant_row['oos_net_pnl_retention_vs_nominal'] * 100.0:.1f}%** du pnl OOS du nominal.",
    f"- Le bucket `high` est maintenant coupé à **{HIGH_BUCKET_MULTIPLIER:.2f}x**.",
]
display(Markdown("\n".join(quick_lines)))


### Synthese executive
- Le nominal OOS fait `35,575.0 USD` avec Sharpe `1.905` et maxDD `-9,765.0 USD`.
- Le 3-state retenu OOS fait `27,959.0 USD` avec Sharpe `3.023` et maxDD `-2,420.5 USD`.
- La variante retenue conserve **78.6%** du pnl OOS du nominal.
- Le bucket `high` est maintenant coupé à **0.25x**.

## 2. Intégrité de la Comparaison

On vérifie ici qu’on compare bien le même trade set de base, avec seulement un changement d’exposition.


In [5]:
integrity_table = pd.DataFrame(
    [
        {"check": "baseline trade count", "value": int(len(baseline_trades))},
        {"check": "retained 3state trade count", "value": int(len(variant_trades))},
        {"check": "matched trades", "value": int(same_trade_rows.shape[0])},
        {"check": "baseline-only rows", "value": int(trade_comparison["quantity_nominal"].notna().sum() - same_trade_rows.shape[0])},
        {"check": "3state-only rows", "value": int(trade_comparison["quantity_3state"].notna().sum() - same_trade_rows.shape[0])},
        {"check": "same exit time", "value": int(same_trade_rows["same_exit_time"].sum())},
        {"check": "same exit reason", "value": int(same_trade_rows["same_exit_reason"].sum())},
        {"check": "same stop price", "value": int(same_trade_rows["same_stop_price"].sum())},
        {"check": "same target price", "value": int(same_trade_rows["same_target_price"].sum())},
    ]
)
display(integrity_table)


,check,value
0,baseline trade count,1134
1,retained 3state trade count,913
2,matched trades,913
3,baseline-only rows,221
4,3state-only rows,0
5,same exit time,913
6,same exit reason,913
7,same stop price,913
8,same target price,913


## 3. Paramètres Exacts

On remet côte à côte le baseline nominal et le mapping effectif du 3-state retenu.


In [6]:
baseline_df = pd.DataFrame([{"parameter": key, "value": value} for key, value in baseline_config.items()])

bucket_display = bucket_map.copy()
for col in ["lower_bound", "upper_bound", "is_composite_score", "effective_risk_per_trade_pct", "oos_net_pnl", "oos_sharpe", "oos_max_drawdown"]:
    if col in bucket_display.columns:
        bucket_display[col] = pd.to_numeric(bucket_display[col], errors="coerce").round(3)

display(Markdown("### Baseline nominal"))
display(baseline_df)
display(Markdown("### Bucket mapping retenu"))
display(bucket_display)


### Baseline nominal

,parameter,value
0,or_minutes,30
1,opening_time,09:30:00
2,direction,both
3,one_trade_per_day,True
4,entry_buffer_ticks,2
5,stop_buffer_ticks,2
6,target_multiple,2.0
7,vwap_confirmation,True
8,vwap_column,continuous_session_vwap
9,time_exit,16:00:00


### Bucket mapping retenu

,bucket_label,bucket_position,lower_bound,upper_bound,risk_multiplier,is_composite_score,oos_n_obs,oos_net_pnl,oos_sharpe,oos_max_drawdown,effective_risk_per_trade_pct
0,low,1,0.337,0.943,0.50,-3.313,141,11704.5,1.968,-9419.5,0.750
1,mid,2,0.943,1.141,1.00,1.622,98,22672.0,5.277,-2439.5,1.500
2,high,3,1.141,1.822,0.75,-3.254,100,1198.5,0.262,-10840.0,0.375


## 4. Tableau Comparatif

On compare ici `nominal` vs `3-state retenu`.


In [7]:
comparison_metrics = pd.DataFrame(
    [
        {
            "variant_name": BASELINE_NAME,
            "overall_net_pnl": baseline_row["overall_net_pnl"],
            "overall_sharpe": baseline_row["overall_sharpe"],
            "overall_profit_factor": baseline_row["overall_profit_factor"],
            "overall_max_drawdown": baseline_row["overall_max_drawdown"],
            "oos_net_pnl": baseline_row["oos_net_pnl"],
            "oos_sharpe": baseline_row["oos_sharpe"],
            "oos_profit_factor": baseline_row["oos_profit_factor"],
            "oos_max_drawdown": baseline_row["oos_max_drawdown"],
        },
        {
            "variant_name": variant_row["variant_name"],
            "overall_net_pnl": variant_row["overall_net_pnl"],
            "overall_sharpe": variant_row["overall_sharpe"],
            "overall_profit_factor": variant_row["overall_profit_factor"],
            "overall_max_drawdown": variant_row["overall_max_drawdown"],
            "oos_net_pnl": variant_row["oos_net_pnl"],
            "oos_sharpe": variant_row["oos_sharpe"],
            "oos_profit_factor": variant_row["oos_profit_factor"],
            "oos_max_drawdown": variant_row["oos_max_drawdown"],
        },
    ]
)
display(comparison_metrics)
display(variant_metrics)


,variant_name,overall_net_pnl,overall_sharpe,overall_profit_factor,overall_max_drawdown,oos_net_pnl,oos_sharpe,oos_profit_factor,oos_max_drawdown
0,nominal,47018.0,0.776227,1.153465,-11763.0,35575.0,1.905281,1.428075,-9765.0
1,sizing_3state_high_0p25,42696.5,1.110109,1.293480,-7915.5,27959.0,3.023167,1.800659,-2420.5


,scope,net_pnl,sharpe,profit_factor,max_drawdown,n_trades,pct_days_traded,worst_day,win_rate
0,overall,42696.5,1.110109,1.293480,-7915.5,913,0.522610,-750.0,0.499452
1,is,14737.5,0.681423,1.133294,-7915.5,656,0.825157,-750.0,0.472561
2,oos,27959.0,3.023167,1.800659,-2420.5,257,0.758112,-744.0,0.568093


## 5. Courbes de Capital

On regarde les courbes full sample et OOS only pour la version retenue.


In [8]:
fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=("Full sample - Equity", "OOS only - Equity", "Full sample - Drawdown", "OOS only - Drawdown"),
    horizontal_spacing=0.1,
    vertical_spacing=0.12,
)

for curve_name, curve_df, color in [
    ("Nominal", baseline_curve, "#2563eb"),
    ("3-state retained", variant_curve, "#16a34a"),
]:
    fig.add_trace(go.Scatter(x=curve_df["session_date"], y=curve_df["equity"], mode="lines", name=f"{curve_name} full", line=dict(width=2.5, color=color)), row=1, col=1)
    fig.add_trace(go.Scatter(x=curve_df["session_date"], y=curve_df["drawdown_usd"], mode="lines", name=f"{curve_name} full DD", showlegend=False, line=dict(width=1.8, color=color, dash="dot")), row=2, col=1)

for curve_name, curve_df, color in [
    ("Nominal", baseline_curve_oos, "#2563eb"),
    ("3-state retained", variant_curve_oos, "#16a34a"),
]:
    fig.add_trace(go.Scatter(x=curve_df["session_date"], y=curve_df["equity"], mode="lines", name=f"{curve_name} oos", line=dict(width=2.5, color=color)), row=1, col=2)
    fig.add_trace(go.Scatter(x=curve_df["session_date"], y=curve_df["drawdown_usd"], mode="lines", name=f"{curve_name} oos DD", showlegend=False, line=dict(width=1.8, color=color, dash="dot")), row=2, col=2)

fig.update_layout(height=820, width=1200, title="MNQ ORB comparison - nominal vs retained sizing_3state (high=0.25)", legend=dict(orientation="h", y=1.08))
fig.show()


## 6. Table Trade par Trade

Cette table montre directement ce que change l’overlay retenu sur les trades matchés.


In [9]:
trade_table = same_trade_rows[
    [
        "session_date",
        "entry_time",
        "direction",
        "quantity_nominal",
        "quantity_3state",
        "risk_multiplier",
        "risk_pct_nominal",
        "risk_pct_3state",
        "net_pnl_nominal",
        "net_pnl_3state",
        "size_ratio_3state_vs_nominal",
        "abs_pnl_ratio_3state_vs_nominal",
    ]
].copy()

for col in ["risk_multiplier", "risk_pct_nominal", "risk_pct_3state", "net_pnl_nominal", "net_pnl_3state", "size_ratio_3state_vs_nominal", "abs_pnl_ratio_3state_vs_nominal"]:
    trade_table[col] = pd.to_numeric(trade_table[col], errors="coerce").round(3)

display(trade_table.head(40))


,session_date,entry_time,direction,quantity_nominal,quantity_3state,risk_multiplier,risk_pct_nominal,risk_pct_3state,net_pnl_nominal,net_pnl_3state,size_ratio_3state_vs_nominal,abs_pnl_ratio_3state_vs_nominal
0,2019-05-07,2019-05-07 14:02:00+00:00,short,6,1.0,0.25,1.5,0.375,345.0,57.5,0.167,0.167
1,2019-05-09,2019-05-09 14:05:00+00:00,short,5,1.0,0.25,1.5,0.375,-680.0,-136.0,0.200,0.200
2,2019-05-13,2019-05-13 14:02:00+00:00,short,5,1.0,0.25,1.5,0.375,345.0,69.0,0.200,0.200
3,2019-05-20,2019-05-20 14:26:00+00:00,long,6,3.0,0.50,1.5,0.750,-333.0,-166.5,0.500,0.500
4,2019-06-03,2019-06-03 14:02:00+00:00,short,4,1.0,0.25,1.5,0.375,478.0,119.5,0.250,0.250
5,2019-06-18,2019-06-18 14:04:00+00:00,long,4,2.0,0.50,1.5,0.750,-360.0,-180.0,0.500,0.500
6,2019-08-02,2019-08-02 14:05:00+00:00,short,7,3.0,0.50,1.5,0.750,238.0,102.0,0.429,0.429
7,2019-08-06,2019-08-06 14:04:00+00:00,long,8,4.0,0.50,1.5,0.750,-748.0,-374.0,0.500,0.500
8,2019-08-07,2019-08-07 14:11:00+00:00,long,4,4.0,1.00,1.5,1.500,792.0,792.0,1.000,1.000
10,2019-08-14,2019-08-14 14:13:00+00:00,short,7,3.0,0.50,1.5,0.750,1407.0,603.0,0.429,0.429


## 7. Distribution et Buckets

On regarde comment la version retenue redistribue l’exposition par bucket.


In [10]:
trade_buckets = variant_trades.copy()
if "bucket_label" not in trade_buckets.columns:
    trade_buckets = trade_buckets.merge(
        variant_controls[["session_date", "phase", "bucket_label"]],
        on="session_date",
        how="left",
    )

bucket_trade_stats = (
    trade_buckets.groupby(["bucket_label", "risk_multiplier"], dropna=False)
    .agg(
        n_trades=("trade_id", "count"),
        avg_quantity=("quantity", "mean"),
        avg_risk_pct=("risk_per_trade_pct", "mean"),
        total_net_pnl_usd=("net_pnl_usd", "sum"),
        avg_net_pnl_usd=("net_pnl_usd", "mean"),
    )
    .reset_index()
    .sort_values(["risk_multiplier", "bucket_label"])
)
for col in ["avg_quantity", "avg_risk_pct", "total_net_pnl_usd", "avg_net_pnl_usd"]:
    bucket_trade_stats[col] = pd.to_numeric(bucket_trade_stats[col], errors="coerce").round(2)
display(bucket_trade_stats)

fig = px.bar(
    bucket_trade_stats,
    x="bucket_label",
    y="total_net_pnl_usd",
    color="risk_multiplier",
    barmode="group",
    title="Contribution PnL par bucket - version retenue",
)
fig.show()


,bucket_label,risk_multiplier,n_trades,avg_quantity,avg_risk_pct,total_net_pnl_usd,avg_net_pnl_usd
0,high,0.25,171,1.01,0.38,-789.5,-4.62
1,low,0.50,379,1.49,0.75,1436.0,3.79
2,mid,1.00,363,3.50,1.50,42050.0,115.84


## 8. Conclusion

Le notebook reflète maintenant la version retenue `high=0.25`.


In [11]:
final_lines = [
    "### Verdict simple",
    f"- Le comparatif affiche maintenant la version retenue: **high = {HIGH_BUCKET_MULTIPLIER:.2f}x**.",
    f"- Le nominal garde la pleine exposition: OOS net `{fmt_money(baseline_row['oos_net_pnl'])}`, Sharpe `{fmt_float(baseline_row['oos_sharpe'])}`.",
    f"- La version retenue réduit plus agressivement le bucket `high`: OOS net `{fmt_money(variant_row['oos_net_pnl'])}`, Sharpe `{fmt_float(variant_row['oos_sharpe'])}`.",
    f"- La lecture correcte reste: **même alpha ORB, exposition plus conservative et plus live-oriented**.",
]
display(Markdown("\n".join(final_lines)))


### Verdict simple
- Le comparatif affiche maintenant la version retenue: **high = 0.25x**.
- Le nominal garde la pleine exposition: OOS net `35,575.0 USD`, Sharpe `1.905`.
- La version retenue réduit plus agressivement le bucket `high`: OOS net `27,959.0 USD`, Sharpe `3.023`.
- La lecture correcte reste: **même alpha ORB, exposition plus conservative et plus live-oriented**.